In [ ]:
#Step-1:Creating a farm dataset

import numpy as np
import pandas as pd

np.random.seed(42)

#Generate synthetic farming data
n=120

rainfall=np.random.uniform(400,1200,n)  #mm
fertilizer=np.random.uniform(50,300,n)  #kg
temperature=np.random.uniform(18,35,n)  #Celsius

#True relationship(unknown to model)
yield_tons=(0.005*rainfall+
            0.03*fertilizer-
            0.1*temperature+
            np.random.normal(0,2,n))    #noise

data=pd.DataFrame({
    "Rainfall":rainfall,
    "Fertilizer":fertilizer,
    "Temperature":temperature,
    "Yield":yield_tons
})
print(data.head())

#Step 2:Split into Train,Validation,Test 70% Train,15%Validation,15%Test

from sklearn.model_selection import train_test_split

X=data[["Rainfall","Fertilizer","Temperature"]]
y=data["Yield"]

#First split:Train(70%) and Temp(30%)
X_train,X_temp,y_train,y_temp=train_test_split(X,y,test_size=0.30,random_state=1)

#Second split:Validation(15%) and Test(15%)
X_val,X_test,y_val,y_test=train_test_split(X_temp,y_temp,test_size=0.50,random_state=1)

print("Training set:",X_train.shape)
print("Validation set:",X_val.shape)
print("Test set:",X_test.shape)

#Step3:Train Regression Model
from sklearn.linear_model import LinearRegression

model=LinearRegression()
model.fit(X_train,y_train)

#Step4:Evaluate Using Regression Metrics

from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score

def evaluate_model(X,y,name):
  y_pred=model.predict(X)
  mae=mean_absolute_error(y,y_pred)
  mse=mean_squared_error(y,y_pred)
  rmse=np.sqrt(mse)
  r2=r2_score(y,y_pred)

  print(f"\n{name}Set Performance")
  print("MAE:",mae)
  print("MSE:",mse)
  print("RMSE:",rmse)
  print("R2 :",r2)

  evaluate_model(X_train,y_train,"Training")
  evaluate_model(X_val,y_val,"Validation")
  evaluate_model(X_test,y_test,"Test")

#Step5:k-Fold Cross Validation (Different k)

from sklearn.model_selection import cross_val_score

def cross_validation_results(k):
  scores=cross_val_score(model,X,y,scoring="neg_mean_squared_error",cv=k)
  rmse_scores=np.sqrt(-scores)

  print(f"\n{k}-Fold Cross Validation Results")
  print("RMSE per fold:",rmse_scores)
  print("Average RMSE:",rmse_scores.mean())
  print("Std Dev RMSE:",rmse_scores.std())

for k in[2,5,10]:
  cross_validation_results(k)


      Rainfall  Fertilizer  Temperature      Yield
0   699.632095  251.860039    33.987796  10.827216
1  1160.571445  274.022825    34.216786   8.126232
2   985.595153  129.500869    33.552695   9.723799
3   878.926787   77.512981    24.292698   0.386578
4   524.814912  106.983791    18.262762   3.703742
Training set: (84, 3)
Validation set: (18, 3)
Test set: (18, 3)

2-Fold Cross Validation Results
RMSE per fold: [1.68180289 2.01102768]
Average RMSE: 1.846415288414677
Std Dev RMSE: 0.16461239377886094

5-Fold Cross Validation Results
RMSE per fold: [1.75692743 1.69460487 1.6286888  1.81498321 2.37968162]
Average RMSE: 1.8549771848539023
Std Dev RMSE: 0.26961221269407826

10-Fold Cross Validation Results
RMSE per fold: [2.19271321 1.15457989 1.46792604 1.88297158 1.22329411 1.90463037
 1.45252255 1.99262521 2.34252822 2.35106136]
Average RMSE: 1.7964852546595433
Std Dev RMSE: 0.42275434659127453
